In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from operator import itemgetter
import random
import sys
import time
from IPython.display import clear_output
import math
import utils
import Conformal
import evaluate
import importlib
importlib.reload(utils)
importlib.reload(evaluate)
importlib.reload(Conformal)


        


ModuleNotFoundError: No module named 'pandas'

In [ ]:
def main(file_name, G, Rounds, K_values):
    
    # --------------------------
    # Load ratings and initialize
    # --------------------------
    start_time = time.time()
    ratings = pd.read_csv(file_name, sep=",", names=["user_id", "item_id", "rating", "timestamp"])
    
    USERS = ratings['user_id'].unique() # assumed that user and item ids are enumerated in the order with no gaps
    ITEMS = ratings['item_id'].unique()
    ITEMS.sort()
    USERS.sort()
    #print(USERS[1], len(USERS))
    nusers = len(USERS)
    nitems = len(ITEMS)
    
    # Initialize profiles
    profiles_train = [np.array([], dtype=int) for _ in USERS]
    profiles_trn_V  = [np.array([], dtype=int) for _ in USERS]

    profiles_test  = [np.array([], dtype=int) for _ in USERS]
    #V              = [np.array([], dtype=int) for _ in USERS]
    
    # Build full profiles per user
    profiles_series = ratings.groupby('user_id')['item_id'].apply(np.array)
    profiles = [profiles_series.get(i, np.array([], dtype=int)) for i in range(nusers)]
    
    # Item support counts
    support = np.bincount(ratings['item_id'], minlength=nitems)
    coCount = utils.coexistenceCount(profiles, nitems)

    
    # --------------------------
    # Initialize evaluation metrics
    # --------------------------
    nK = len(K_values)

    pf_score = np.zeros((6, nK))   # Avg values for 6 evaluation metrics
    pf_score1 = np.zeros(8)
    baseline_time = 0.0
    CRS_time = 0.0
    pre_time = 0.0
    cmn_time = 0.0



    pf_score[0] = np.zeros(nK)
    pf_score[1] = np.zeros(nK)
    pf_score[2]  = np.zeros(nK)
    pf_score[3] = np.zeros(nK)
    pf_score[4] = np.zeros(nK)
    pf_score[5]  = np.zeros(nK)
    
    # --------------------------
    # Train / Validation / Test split
    # --------------------------
    for user in USERS:
        prof = np.array(profiles[user], dtype=int)
        np.random.shuffle(prof)
        N = len(prof)
        Ntr = N * 60 // 100 # No. of training 
        profiles_trn_V[user] = prof[: Ntr]
        profiles_test[user]  = prof[Ntr:]
        
    
    # --------------------------
    # Evaluation rounds
    # --------------------------
    run = 0

    n_groups = Rounds    # number of groups to form


    ###### Random Grouping ######
    np.random.shuffle(USERS)

    # Select only as many users as needed
    total_needed = n_groups * G
    selected_users = USERS[:total_needed]

    # Form #n_groups of size k each
    # groups = [selected_users[i:i+G] for i in range(0, total_needed, G)]
    # Just select individual users
    # user = selected_users[run]

    pre_time1 = time.time() - start_time

    while run < Rounds:
        
        start_time = time.time()
        # group = groups[run]
        user = selected_users[run]
        # --------------------------
        # Build group items
        # --------------------------
        # group_items = np.array(list(set().union(*(profiles_trn_V[u] for u in group))))
        user_items = profiles_trn_V[user]
        #snp.random.shuffle(group_items)

        Ng =  len(user_items)
        if  Ng <= 60 :
            group_train = user_items[: (Ng*75//100)]
            group_V     = user_items[(Ng*75//100) : ]
        else:
            k1=15
            group_train  = user_items[:Ng-k1] 
            group_V      = user_items[Ng-k1: ]
            #profiles_test[user]  = prof[Ntr :]
        
        #group_V     = np.array(list(set().union(*(V[u] for u in group))))
        #group_items = np.concatenate((group_train, group_V))
        cand_items =  np.setdiff1d(ITEMS, user_items)
        #print(group_train.shape, group_V.shape)
        # for u in group:
        #      profiles_test[u] = np.setdiff1d(profiles_test[u], group_items, assume_unique=True)
        profiles_test[user] = np.setdiff1d(profiles_test[user], user_items, assume_unique=True)

        support1, coCount1 = utils.modify(user, -1, profiles_train, profiles_test, support, coCount)
        #support1 = support
        #coCount1 = coCount
        
        # weights = utils.Virtual_user_weights(profiles_train, user, user_items, G, support1, coCount1, nusers)
        weights = utils.Personal_user_weights(profiles_train, user, user_items, support1, coCount1, nusers)
        n = len(group_train)
        train_weights = weights[:n]
        pre_time += time.time() - start_time

        #V_weights = weights[n:]
        #profile_items = np.array(group_items, dtype=int)
        
        # --------------------------
         # Candidate recommendations
        # --------------------------
        start_time = time.time()

        mask = np.ones(nitems, dtype=bool)  
        mask[user_items] = False                  # exclude group items
        mask &= (support > 0)                      # keep only items with support > 0
        cand_items = ITEMS[mask]

        max_item = int(max(cand_items))  # get the max index

        start_time = time.time()

        scorep = np.zeros(max_item + 1)  # initialize score array

        # Fill array: use item ID as index
        for item in cand_items:
            scorep[item] = utils.scoreR(user_items, item, weights, support1, coCount1, nusers)

        rec_cand_sorted = np.array(\
             sorted(cand_items, key=lambda x: scorep[x], reverse=True))

        # Select top-K recommendations
        #recommendations = dict(list(rec_candidates.items())[:top_K])
        # --------------------------
        # Baseline GRS
        # --------------------------
        # metrics1 = evaluate.evaluate_group_ranking_metrics(user, profiles_test, rec_cand_sorted, K_values)
        metrics1 = evaluate.evaluate_personal_ranking_metrics(user, profiles_test, rec_cand_sorted, K_values)
        baseline_time += time.time() - start_time


        # --------------------------
        # Conformal GRS
        # --------------------------
        start_time = time.time()

        k = len(group_V)

        p = np.zeros(len(cand_items), dtype=float)

        p = np.zeros(len(cand_items), dtype=float)

        for v in group_V:
            coexist = coCount1[group_train, v]     # shape: (len(group_train),)
            wt_sup = train_weights * coexist      # shape: (len(group_train),)
            # Compare wt_sup with coCount[item, v] for all candidate items
            co_v = coCount1[cand_items, v]              # shape: (len(cand_items),)
            # Broadcasting comparison:    # (len(cand_items), len(group_train))
            comp = wt_sup <= (np.max(train_weights)-0.05)*co_v[:, None]              # Compare each candidate vs group
            p += (comp.sum(axis=1)+1) / (n+1)                  # Sum over group dimension

        p /= k
    
        sorted_indices = np.argsort(-p)
        sorted_cand_items = cand_items[sorted_indices]
        #sorted_cand_items = Conformal.conformal(cand_items, group_train, group_V, train_weights, coCount, n)
        
        # metrics2 = evaluate.evaluate_group_ranking_metrics(user, profiles_test, sorted_cand_items, K_values)
        metrics2 = evaluate.evaluate_personal_ranking_metrics(user, profiles_test, sorted_cand_items, K_values)
        CRS_time += time.time() - start_time

        if not metrics1 or not metrics2:
            continue
        # --------------------------
        # GRS results
        # --------------------------
        pf_score[0] += np.array([metrics1[f"precision@{K}"] for K in K_values])
        pf_score[1] += np.array([metrics1[f"recall@{K}"] for K in K_values])
        pf_score[2] += np.array([metrics1[f"f1@{K}"] for K in K_values])
        pf_score1[0] += metrics1["AUC"]
        pf_score1[1] += metrics1["nDCG"]
        pf_score1[2] += metrics1["ap"]
        pf_score1[3] +=metrics1["rr"]

       # --------------------------
        # Conformal GRS results
        # --------------------------
        pf_score[3] += np.array([metrics2[f"precision@{K}"] for K in K_values])
        pf_score[4] += np.array([metrics2[f"recall@{K}"] for K in K_values])
        pf_score[5] += np.array([metrics2[f"f1@{K}"] for K in K_values])
        pf_score1[4] += metrics2["AUC"]
        pf_score1[5] += metrics2["nDCG"]
        pf_score1[6] += metrics2["ap"]
        pf_score1[7] += metrics2["rr"]
        
        # --------------------------
        # Update round
        # --------------------------
        run += 1
        clear_output(wait=True)
        print(f"Round {run} completed")
    
    # --------------------------
    # Average metrics over rounds
    # --------------------------
    pf_score /= Rounds
    pf_score1 /= Rounds
    baseline_time =  (baseline_time+pre_time)/Rounds
    CRS_time =  (CRS_time+pre_time)/Rounds    
    return (pf_score, pf_score1), baseline_time, CRS_time, pre_time1


In [ ]:
file_name = "Data/ml_100k_rating1.csv"
# file_name = "Data/ml_latest_small_rating1.csv"
# file_name = "Data/personality_2018_rating1.csv"
# file_name = "Data/ml_1m_rating1.csv"
# file_name = "Data/ml_10m_rating1.csv"
# file_name = "Data/ml_20m_rating1.csv" 
# file_name = "Data/ml_25m_rating1.csv" 
# file_name = "Data/ml_latest_rating1.csv" 

K_values = [1,2,5,10, 15, 20, 40]
Gsize = 2
Rounds = 100
(pf_score, pf_score1), baseline_time, CRS_time, trn_time = main(file_name, Gsize, Rounds, K_values)


Round 100 completed


In [4]:
res1 = {
        "K": K_values,
        "Precision (GRS)": pf_score[0],
        "Precision (CGRS)": pf_score[3],
        "Recall (GRS)": pf_score[1],
        "Recall (CGRS)": pf_score[4],
        "F1 (GRS)": pf_score[2],
        "F1 (CGRS)": pf_score[5],
    }
res_df = pd.DataFrame(res1)
print(res_df.to_string(index=False, float_format="%.4f"))
res2 = {
    "Metric": ["AUC", "nDCG", "AP", "RR"],
    "GRS": pf_score1[0:4],
    "CGRS": pf_score1[4:8],
}

df = pd.DataFrame(res2)
print(df.to_string(index=False, float_format="%.4f"))
#print(pf_score, pf_score1)

print(f"GRS_time:{baseline_time}\n")
print(f"CGRS_time:{CRS_time}\n")
print(f"Preprocessing time:{trn_time}\n")
print(f"GRS_tot_time: {trn_time+baseline_time}\n")
print(f"CGRS_tot_time: {trn_time+CRS_time}\n") 

with open("results_rand_time.txt", "a") as f:
    f.write("\n"+"-" * 40 + "\n")  # separator line
    f.write(f"File: {file_name}, group_size:{Gsize}, Rounds:{Rounds}, Prof_size: 20-40, Mean-P, weight:max-0.05 \n")
    f.write("-" * 40 + "\n")  # separator line
    f.write(res_df.to_string(index=False, float_format="%.4f")+"\n")
    f.write("-" * 40 + "\n")  # separator line
    f.write(df.to_string(index=False, float_format="%.4f")+"\n")
    f.write("-" * 40 + "\n")  # separator line
    f.write(f"GRS_time_test: {baseline_time}\n")
    f.write(f"CGRS_time_test: {CRS_time}\n") 
    f.write(f"Preprocessing time:{trn_time}\n")
    f.write(f"GRS_tot_time: {trn_time+baseline_time}\n")
    f.write(f"CGRS_tot_time: {trn_time+CRS_time}\n") 
    f.write("-" * 40 + "\n")  # separator line

 K  Precision (GRS)  Precision (CGRS)  Recall (GRS)  Recall (CGRS)  F1 (GRS)  F1 (CGRS)
 1           0.3250            0.3500        0.0112         0.0110    0.0210     0.0207
 2           0.2925            0.3150        0.0194         0.0194    0.0348     0.0348
 5           0.2560            0.2820        0.0408         0.0443    0.0642     0.0690
10           0.2085            0.2410        0.0628         0.0735    0.0831     0.0960
15           0.1863            0.2197        0.0821         0.0981    0.0948     0.1127
20           0.1755            0.2040        0.1045         0.1186    0.1064     0.1218
40           0.1509            0.1684        0.1711         0.1856    0.1259     0.1392
Metric    GRS   CGRS
   AUC 0.9840 0.9856
  nDCG 0.4951 0.5112
    AP 0.1103 0.1310
    RR 0.4555 0.4791
GRS_time:1.336372847557068

CGRS_time:10.955727829933167

Preprocessing time:1766.1415376663208

GRS_tot_time: 1767.4779105138778

CGRS_tot_time: 1777.097265496254

